## Datos de caminata
[link](https://archive.ics.uci.edu/dataset/240/human%2Bactivity%2Brecognition%2Busing%2Bsmartphones)

In [153]:
import numpy as np
import pandas as pd
import time

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

from sklearn.preprocessing import StandardScaler

In [154]:
MAPA_ACTIVIDADES = {
    1: "Caminando",
    2: "Caminando hacia arriba",
    3: "Caminando hacia abajo",
    4: "Sentado",
    5: "Parado",
    6: "Acostado"
}

## CARGAR DATASET PRINCIPAL

In [155]:
datos = np.load(
    "datos/har_dataset.npz"
)

X_train = datos["X_train"]
y_train = datos["y_train"]

X_test = datos["X_test"]
y_test = datos["y_test"]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# CARGAR DATOS INDIVIDUALES

datos_individual = np.load(
    "datos/har_dataset_individual.npz"
)

X_individual = datos_individual["X_individual"]
y_individual = datos_individual["y_individual"]

indices_individuales = datos_individual[
    "indices_individuales"
]

def nombre_actividad(etiqueta):
    return MAPA_ACTIVIDADES[int(etiqueta)]


print("Datos cargados correctamente:\n")

print(
    f"X_train:      {X_train.shape} | "
    f"y_train:      {y_train.shape}"
)

print(
    f"X_test:       {X_test.shape} | "
    f"y_test:       {y_test.shape}"
)

print(
    f"X_individual: {X_individual.shape} | "
    f"y_individual: {y_individual.shape}"
)

print(
    f"Índices individuales: "
    f"{indices_individuales}"
)


print("\nActividades:")

for numero, nombre in MAPA_ACTIVIDADES.items():
    print(f"{numero}: {nombre}")


Datos cargados correctamente:

X_train:      (7352, 561) | y_train:      (7352,)
X_test:       (2947, 561) | y_test:       (2947,)
X_individual: (10, 561) | y_individual: (10,)
Índices individuales: [ 253  262  277  593 1273 1290 1924 2053 2274 2526]

Actividades:
1: Caminando
2: Caminando hacia arriba
3: Caminando hacia abajo
4: Sentado
5: Parado
6: Acostado


### FUNCION PARA METRICAS

In [156]:
def calcular_metricas(y_real, y_pred):

    return {

        "accuracy":
            accuracy_score(
                y_real,
                y_pred
            ),

        "precision":
            precision_score(
                y_real,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "recall":
            recall_score(
                y_real,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "f1_score":
            f1_score(
                y_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
    }

## MLP

In [157]:
modelo_mlp = MLPClassifier(

    hidden_layer_sizes=(
        128,
        64
    ),

    activation="relu",

    solver="adam",

    max_iter=100,

    early_stopping=True,

    validation_fraction=0.1,

    n_iter_no_change=10,

    random_state=42
)

### Entrenamiento

In [158]:
inicio = time.perf_counter()

modelo_mlp.fit(
    X_train,
    y_train
)

tiempo_entrenamiento_mlp = (
    time.perf_counter() - inicio
)

### Prediccion

In [159]:
inicio = time.perf_counter()

y_pred_mlp = modelo_mlp.predict(
    X_test
)

tiempo_prediccion_mlp = (
    time.perf_counter() - inicio
)


### Metricas

In [160]:
metricas_mlp = calcular_metricas(
    y_test,
    y_pred_mlp
)

matriz_mlp = confusion_matrix(
    y_test,
    y_pred_mlp
)


print("\nResultados MLP:")

for nombre, valor in metricas_mlp.items():

    print(
        f"{nombre}: {valor:.4f}"
    )

print(
    f"Tiempo entrenamiento: "
    f"{tiempo_entrenamiento_mlp:.4f} s"
)

print(
    f"Tiempo predicción: "
    f"{tiempo_prediccion_mlp:.4f} s"
)


Resultados MLP:
accuracy: 0.9474
precision: 0.9476
recall: 0.9474
f1_score: 0.9474
Tiempo entrenamiento: 2.6674 s
Tiempo predicción: 0.0043 s


## CNN

In [161]:
X_train_cnn = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_cnn = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

In [162]:
modelo = tf.keras.Sequential([

    tf.keras.layers.Conv1D(
        filters=32,
        kernel_size=5,
        activation="relu",
        input_shape=(561, 1)
    ),

    tf.keras.layers.MaxPooling1D(
        pool_size=2
    ),

    tf.keras.layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu"
    ),

    tf.keras.layers.MaxPooling1D(
        pool_size=2
    ),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        6,
        activation="softmax"
    )
])


modelo.compile(
optimizer=tf.keras.optimizers.Adam(
learning_rate=0.01
),
loss="sparse_categorical_crossentropy",
metrics=["accuracy"]
)


modelo.summary()

c:\Users\Jpose75\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_16 (Conv1D)              │ (None, 557, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_16 (MaxPooling1D) │ (None, 278, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 274, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_17 (MaxPooling1D) │ (None, 137, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 8768)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │       561,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 572,102 (2.18 MB)

 Trainable params: 572,102 (2.18 MB)

 Non-trainable params: 0 (0.00 B)

### ENTRENAR

In [163]:
inicio = time.perf_counter()

historial = modelo.fit(
    X_train_cnn,
    y_train - 1,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

tiempo_entrenamiento_cnn = time.perf_counter() - inicio

Epoch 1/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.4850 - loss: 1.7113 - val_accuracy: 0.8356 - val_loss: 0.5018
Epoch 2/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.8294 - loss: 0.4383 - val_accuracy: 0.9171 - val_loss: 0.1892
Epoch 3/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9031 - loss: 0.2347 - val_accuracy: 0.9457 - val_loss: 0.1371
Epoch 4/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9327 - loss: 0.1740 - val_accuracy: 0.9497 - val_loss: 0.1168
Epoch 5/5
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9450 - loss: 0.1436 - val_accuracy: 0.9293 - val_loss: 0.1776


### PREDICCION

In [164]:
inicio = time.perf_counter()

predicciones = modelo.predict(
    X_test_cnn,
    verbose=0
)

tiempo_prediccion_cnn = time.perf_counter() - inicio

In [165]:
y_pred_cnn = np.argmax(
    predicciones,
    axis=1
) + 1

### METRICAS

In [166]:
metricas_cnn = calcular_metricas(
    y_test,
    y_pred_cnn
)

matriz_cnn = confusion_matrix(
    y_test,
    y_pred_cnn
)

print("\nResultados CNN:")

for nombre, valor in metricas_cnn.items():

    print(
        f"{nombre}: {valor:.4f}"
    )

print(
    f"Tiempo entrenamiento: "
    f"{tiempo_entrenamiento_cnn:.4f} s"
)

print(
    f"Tiempo predicción: "
    f"{tiempo_prediccion_cnn:.4f} s"
)


Resultados CNN:
accuracy: 0.9152
precision: 0.9230
recall: 0.9152
f1_score: 0.9152
Tiempo entrenamiento: 5.5039 s
Tiempo predicción: 0.3516 s


## RESULTADOS GENERALES

In [167]:
resultados_generales_1 = pd.DataFrame([

    {

        "modelo": "MLP",

        "accuracy":
            metricas_mlp[
                "accuracy"
            ],

        "precision":
            metricas_mlp[
                "precision"
            ],

        "recall":
            metricas_mlp[
                "recall"
            ],

        "f1_score":
            metricas_mlp[
                "f1_score"
            ],

        "tiempo_entrenamiento_s":
            tiempo_entrenamiento_mlp,

        "tiempo_prediccion_s":
            tiempo_prediccion_mlp,

        "iteraciones":
            modelo_mlp.n_iter_
    },

    {

        "modelo": "CNN",

        "accuracy":
            metricas_cnn[
                "accuracy"
            ],

        "precision":
            metricas_cnn[
                "precision"
            ],

        "recall":
            metricas_cnn[
                "recall"
            ],

        "f1_score":
            metricas_cnn[
                "f1_score"
            ],

        "tiempo_entrenamiento_s":
            tiempo_entrenamiento_cnn,

        "tiempo_prediccion_s":
            tiempo_prediccion_cnn,

        "iteraciones":
            len(
                historial_cnn.history[
                    "loss"
                ]
            )
    }
])

In [168]:
print(
    resultados_generales_1.to_string(
        index=False
    )
)


print("\nMatriz de confusión MLP:")

print(matriz_mlp)


print("\nMatriz de confusión CNN:")

print(matriz_cnn)

modelo  accuracy  precision   recall  f1_score  tiempo_entrenamiento_s  tiempo_prediccion_s  iteraciones
   MLP  0.947404   0.947646 0.947404  0.947410                2.667353             0.004284           46
   CNN  0.915168   0.923044 0.915168  0.915164                5.503935             0.351569           15

Matriz de confusión MLP:
[[482   1  13   0   0   0]
 [ 29 439   3   0   0   0]
 [  4  20 396   0   0   0]
 [  0   3   0 455  33   0]
 [  1   0   0  45 486   0]
 [  0   0   0   0   3 534]]

Matriz de confusión CNN:
[[492   0   4   0   0   0]
 [ 45 422   4   0   0   0]
 [ 75  12 333   0   0   0]
 [  0   1   0 459  24   7]
 [  0   2   0  74 455   1]
 [  0   0   0   1   0 536]]


## PRUEBA INDIVIDUAL

In [169]:
X_individual_cnn = X_individual.reshape(
    -1,
    561,
    1
)

In [170]:
resultados_individuales = []


for i in range(
    len(X_individual)
):

    etiqueta_real = int(
        y_individual[i]
    )

    actividad_real = (
        nombre_actividad(
            etiqueta_real
        )
    )


    # ==================
    # MLP

    muestra_mlp = (
        X_individual[i:i+1]
    )


    inicio = time.perf_counter()

    pred_mlp = int(
        modelo_mlp.predict(
            muestra_mlp
        )[0]
    )

    tiempo_mlp = (
        time.perf_counter() - inicio
    )


    actividad_mlp = (
        nombre_actividad(
            pred_mlp
        )
    )


    # ======================================
    # CNN

    muestra_cnn = (
        X_individual_cnn[i:i+1]
    )


    inicio = time.perf_counter()

    probabilidades = (
        modelo_cnn.predict(
            muestra_cnn,
            verbose=0
        )[0]
    )

    tiempo_cnn = (
        time.perf_counter() - inicio
    )


    pred_cnn = (
        int(
            np.argmax(
                probabilidades
            )
        ) + 1
    )


    actividad_cnn = (
        nombre_actividad(
            pred_cnn
        )
    )


    confianza_cnn = float(
        probabilidades[
            pred_cnn - 1
        ]
    )

    # GUARDAR
    
    resultados_individuales.append({

        "indice_original":
            int(
                indices_individuales[i]
            ),

        # REAL
        "etiqueta_real":
            etiqueta_real,

        "actividad_real":
            actividad_real,

        # MLP
        "prediccion_MLP":
            pred_mlp,

        "actividad_MLP":
            actividad_mlp,

        "correcto_MLP":
            pred_mlp == etiqueta_real,

        "tiempo_prediccion_MLP_s":
            tiempo_mlp,

        # CNN
        "prediccion_CNN":
            pred_cnn,

        "actividad_CNN":
            actividad_cnn,

        "confianza_CNN":
            confianza_cnn,

        "correcto_CNN":
            pred_cnn == etiqueta_real,

        "tiempo_prediccion_CNN_s":
            tiempo_cnn
    })


In [171]:
resultados_individuales_2 = pd.DataFrame(
    resultados_individuales
)

## GUARDAR RESULTADOS

In [172]:
resultados_generales_1.to_csv(
    "resultados/resultados_generales_2.csv",
    index=False
)

resultados_individuales_2.to_csv(
    "resultados/resultados_individuales_2.csv",
    index=False
)


print("\nResultados guardados correctamente.")


Resultados guardados correctamente.
